# 01 — Veri hazırlama ve MFCC
KWS (keyword spotting) PoC — STM32F407 bitirme projesi

Akış: ses (16000 örnek) → doldurma + normalize → MFCC (49×10) → veri seti (`kws_mfcc.npz`)

**Not:** Hücreleri yukarıdan aşağı sırayla çalıştır. İndirme hücresini (1) oturum başına bir kez çalıştırman yeterli.

## 0. Kütüphaneler

In [ ]:
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from scipy.io import wavfile

## 1. Veri setini indir (mini_speech_commands, ~200 MB)

In [ ]:
!wget -q -nc http://storage.googleapis.com/download.tensorflow.org/data/mini_speech_commands.zip
!unzip -qo mini_speech_commands.zip -d data

data_dir = pathlib.Path('data/mini_speech_commands')
labels = sorted(p.name for p in data_dir.iterdir() if p.is_dir())
print(labels)

## 2. Veriyi tanı: kayıtlar hep 1 saniye mi?

In [ ]:
lengths = np.array([len(wavfile.read(f)[1]) for f in data_dir.glob('*/*.wav')])
print("toplam dosya:", len(lengths))
print("en kısa:", lengths.min(), "en uzun:", lengths.max())
print("16000'den kısa olan:", (lengths < 16000).sum())

## 3. Ön işleme: int16 → [-1, 1] float, kısa kayıtları 16000'e tamamla

In [ ]:
    def load_wav(path, target_len=16000):
        _, x = wavfile.read(path)
        x = x.astype(np.float32) / 32768.0           # int16 -> [-1, 1]
        if len(x) < target_len:
            x = np.pad(x, (0, target_len - len(x)))  # sonunu sessizlikle doldur
        return x[:target_len]

    short = next(f for f in data_dir.glob('*/*.wav') if len(wavfile.read(f)[1]) < 16000)
    x = load_wav(short)
    print(short.parent.name, x.shape, x.dtype, x.min(), x.max())
    plt.plot(x); plt.title(short.parent.name); plt.show()

## 4. MFCC
40 ms pencere, 20 ms kaydırma → 49 pencere. FFT 1024 (kartta 2'nin kuvveti gerekiyor). 40 mel filtresi, ilk 10 katsayı → **(49, 10)**

In [ ]:
    def mfcc(x, sr=16000):
        stft = tf.signal.stft(x, frame_length=640, frame_step=320, fft_length=1024)
        spec = tf.abs(stft)                                        # (49, 513)
        mel_w = tf.signal.linear_to_mel_weight_matrix(
            num_mel_bins=40, num_spectrogram_bins=513, sample_rate=sr,
            lower_edge_hertz=20.0, upper_edge_hertz=4000.0)
        mel = tf.tensordot(spec, mel_w, 1)                         # (49, 40)
        log_mel = tf.math.log(mel + 1e-6)
        return tf.signal.mfccs_from_log_mel_spectrograms(log_mel)[:, :10].numpy()  # (49, 10)

    x = load_wav(short)
    m = mfcc(x)
    print(x.dtype, m.shape)
    plt.imshow(m.T, aspect='auto', origin='lower'); plt.colorbar()
    plt.title('MFCC - ' + short.parent.name); plt.xlabel('pencere (20 ms)'); plt.ylabel('katsayı')
    plt.show()

## 5. Bütün veri setini MFCC'ye çevir ve kaydet (birkaç dakika sürer)

In [ ]:
X, y = [], []
for i, lab in enumerate(labels):
    for f in (data_dir / lab).glob('*.wav'):
        X.append(mfcc(load_wav(f)))
        y.append(i)

X = np.array(X, dtype=np.float32)
y = np.array(y)
print(X.shape, y.shape, np.bincount(y))
np.savez('data/kws_mfcc.npz', X=X, y=y, labels=labels)